# 02 · Vision Transformers

> **Source notes:** `VisionTransformers.md`

How does a ViT turn a `(3, 224, 224)` image tensor into a sequence a transformer can process?

This notebook builds it from scratch:
- **Patch extraction** — split an image into 16×16 patches manually
- **Patch embedding** — project each patch to a 768-dim vector
- **Positional embeddings** — add learned position information
- **Attention visualisation** — show which patches attend to which
- **PixelSmith v1** — run a real ViT-B/16 on our synthetic image

**All local. No GPU needed.** Runtime: < 1 minute.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)
print("Packages ready.")

## 1 · Manual Patch Extraction

A ViT splits the image into non-overlapping $P \times P$ patches.

For a 224×224 image with $P = 16$: $N = (224/16)^2 = 196$ patches.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Build the same synthetic test image from Ch.1
H, W, C = 224, 224, 3
img_np = np.zeros((H, W, C), dtype=np.float32)
img_np[:, :, 0] = np.linspace(0, 1, W)[np.newaxis, :]
img_np[:, :, 1] = np.linspace(0, 1, H)[:, np.newaxis]
img_np[:, :, 2] = 0.5

# To PyTorch tensor (C, H, W), add batch dim → (1, C, H, W)
img = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0)  # (1, 3, 224, 224)
print(f"Input image tensor: {img.shape}")

# ── Manual patch extraction using unfold ─────────────────────────────────────
P = 16   # patch size

# unfold along height and width dimensions
patches = img.unfold(2, P, P).unfold(3, P, P)  # (1, 3, 14, 14, 16, 16)
print(f"After unfold: {patches.shape}  (batch, channels, h_patches, w_patches, patch_h, patch_w)")

# Reshape to (1, N, 3*P*P) — sequence of flat patch vectors
N = (H // P) * (W // P)  # 196
patches_flat = patches.contiguous().view(1, C, N, P*P).permute(0, 2, 1, 3).reshape(1, N, C*P*P)
print(f"Flattened patches: {patches_flat.shape}  (batch=1, N=196 patches, patch_dim={C*P*P})")
print(f"Each patch contains {C*P*P} values ({C} channels × {P}×{P} pixels)")

In [ ]:
# Visualise: show the image divided into its 196 patches as a grid
grid_h, grid_w = 14, 14  # 14×14 = 196 patches

fig, axes = plt.subplots(grid_h, grid_w, figsize=(14, 14))
patch_idx = 0
for row in range(grid_h):
    for col in range(grid_w):
        patch = patches_flat[0, patch_idx].reshape(C, P, P).permute(1, 2, 0).numpy()
        axes[row, col].imshow(patch, vmin=0, vmax=1)
        axes[row, col].axis("off")
        patch_idx += 1

plt.suptitle(f"224×224 image split into {N} patches of {P}×{P}\n"
             "Each patch = one token in the ViT sequence", y=1.01)
plt.tight_layout()
plt.show()

## 2 · Building a Minimal ViT Patch Embedding Layer

The patch embedding is a single `Conv2d(in_channels=C, out_channels=d, kernel_size=P, stride=P)`. This is mathematically equivalent to the linear projection — it just uses optimised GPU kernels.

In [ ]:
class PatchEmbedding(nn.Module):
    """ViT patch embedding: image → sequence of patch vectors."""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        
        # Conv2d with kernel=patch_size, stride=patch_size = non-overlapping patch projection
        self.projection = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        # x: (N, C, H, W)
        x = self.projection(x)      # (N, d_model, H/P, W/P)
        x = x.flatten(2)            # (N, d_model, n_patches)
        x = x.transpose(1, 2)       # (N, n_patches, d_model)
        return x


d_model = 768  # ViT-B hidden dimension
patch_embed = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, d_model=d_model)

patch_embeddings = patch_embed(img)  # (1, 196, 768)
print(f"Patch embeddings: {patch_embeddings.shape}")
print(f"  Batch: 1, Sequence length: 196 patches, Embedding dim: {d_model}")
print(f"  Projection layer params: {sum(p.numel() for p in patch_embed.parameters()):,}")
print(f"  (= 768 × (3×16×16 + 1) = 768 × 769 = {768 * 769:,})")

## 3 · CLS Token and Positional Embeddings

In [ ]:
N_PATCHES   = 196
SEQ_LEN     = N_PATCHES + 1  # +1 for [CLS] token

# Learnable [CLS] token — initialised to zeros, learned during training
cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

# Learnable positional embeddings — one per position in the full sequence
pos_embed = nn.Parameter(torch.zeros(1, SEQ_LEN, d_model))

# Prepend CLS to patch sequence
cls_expanded = cls_token.expand(1, -1, -1)       # (1, 1, 768)
seq = torch.cat([cls_expanded, patch_embeddings], dim=1)  # (1, 197, 768)

# Add positional embeddings
seq_with_pos = seq + pos_embed  # (1, 197, 768) — broadcast over batch

print(f"After CLS prepend:        {seq.shape}")
print(f"After positional embed:   {seq_with_pos.shape}")
print(f"\nSequence ready for transformer encoder:")
print(f"  Position 0:     [CLS] token (learnable aggregator)")
print(f"  Positions 1-196: patch embeddings (row-major: top-left to bottom-right)")
print(f"\nPositional embedding params: {SEQ_LEN} × {d_model} = {SEQ_LEN * d_model:,}")

## 4 · Self-Attention Over Patches — Minimal Implementation

Standard scaled dot-product attention, now applied to patch embeddings instead of text tokens.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """Q, K, V: (batch, heads, seq_len, d_k)"""
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)  # (b, h, seq, seq)
    weights = F.softmax(scores, dim=-1)                             # attention weights
    return torch.matmul(weights, V), weights


# Single-head attention on a small sequence for illustration
SEQ, D = 10, 64  # 10 patches, 64-dim embeddings
x_small = torch.randn(1, SEQ, D)  # small batch for clarity

Wq = nn.Linear(D, D, bias=False)
Wk = nn.Linear(D, D, bias=False)
Wv = nn.Linear(D, D, bias=False)

Q = Wq(x_small).unsqueeze(1)  # (1, 1, 10, 64)
K = Wk(x_small).unsqueeze(1)
V = Wv(x_small).unsqueeze(1)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Attention input:   {x_small.shape}")
print(f"Attention weights: {attn_weights.shape}  (batch, heads, seq_len, seq_len)")
print(f"Attention output:  {output.squeeze(1).shape}")

# Visualise attention map
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(attn_weights[0, 0].detach().numpy(), cmap="viridis", aspect="auto")
ax.set_xlabel("Key (source patch)")
ax.set_ylabel("Query (target patch)")
ax.set_title(f"Attention weights ({SEQ}×{SEQ})\n"
             "A[i,j] = how much patch i attends to patch j")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f"\nEach row sums to 1: {attn_weights[0,0].sum(dim=-1).detach().round(decimals=4).tolist()}")

## 5 · PixelSmith v1 — Real ViT-B/16 via torchvision

Run the full pretrained ViT-B/16 on our synthetic image to get a real 768-dim embedding.

In [ ]:
import torchvision.models as models
import torchvision.transforms as T

# Load pretrained ViT-B/16 (downloads ~330 MB on first run)
print("Loading ViT-B/16 (pretrained on ImageNet-21k)...")
vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
vit.eval()

total_params = sum(p.numel() for p in vit.parameters())
print(f"Total parameters: {total_params / 1e6:.1f}M")

# ImageNet normalisation transform
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# Use a real photograph from torchvision's example images
from PIL import Image
import urllib.request, io

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
try:
    with urllib.request.urlopen(url, timeout=5) as r:
        img_real = Image.open(io.BytesIO(r.read())).convert("RGB")
    print(f"Downloaded cat image: {img_real.size}")
except Exception:
    # Fallback: use our synthetic gradient image
    img_real = Image.fromarray((img_np * 255).astype(np.uint8))
    print("Using synthetic gradient image (network unavailable)")

img_tensor = transform(img_real).unsqueeze(0)  # (1, 3, 224, 224)
print(f"Input tensor: {img_tensor.shape}")

In [ ]:
with torch.no_grad():
    # ── Extract the CLS token embedding (before the classification head) ────
    # torchvision's ViT: forward_features returns the pre-head features
    features = vit._process_input(img_tensor)   # patch embeddings (1, 196, 768)
    n = features.shape[0]
    cls = vit.class_token.expand(n, -1, -1)      # (1, 1, 768)
    seq = torch.cat([cls, features], dim=1)      # (1, 197, 768)
    seq = seq + vit.encoder.pos_embedding         # add pos embeddings
    encoded = vit.encoder.layers(seq)             # 12 transformer layers
    encoded = vit.encoder.ln(encoded)             # layer norm
    cls_embedding = encoded[:, 0]                 # CLS token → (1, 768)

print(f"Image embedding shape:  {cls_embedding.shape}")
print(f"Embedding norm:         {cls_embedding.norm().item():.4f}")
print(f"Value range:            [{cls_embedding.min().item():.4f}, {cls_embedding.max().item():.4f}]")
print(f"\nFirst 8 embedding values: {cls_embedding[0, :8].tolist()}")
print("\nThis 768-dimensional vector is what CLIP stores and compares with text embeddings (Ch.3).")

In [ ]:
# Verify classification still works (just to check the model loaded correctly)
with torch.no_grad():
    logits = vit(img_tensor)
    top5_idx = logits.topk(5).indices[0].tolist()

# ImageNet class names (a small subset for illustration)
imagenet_labels = {
    281: "tabby cat", 282: "tiger cat", 283: "Persian cat",
    284: "Siamese cat", 285: "Egyptian cat", 287: "lynx",
    463: "bucket", 508: "computer keyboard",
}
print("Top-5 predicted classes:")
for idx in top5_idx:
    label = imagenet_labels.get(idx, f"class_{idx}")
    score = logits[0, idx].item()
    print(f"  [{idx:4d}] {label:<25} logit={score:.3f}")

## 6 · Positional Embedding Similarity

ViT's positional embeddings learn geometric structure from data. The cosine similarity between adjacent positions should be higher than between distant positions.

In [ ]:
with torch.no_grad():
    # pos_embedding has shape (1, 197, 768)
    # Position 0 = CLS, positions 1-196 = patches in row-major order (14×14 grid)
    pos = vit.encoder.pos_embedding[0, 1:]  # (196, 768) — patches only
    pos_norm = F.normalize(pos, dim=-1)
    
    # Similarity matrix: (196, 196)
    sim = (pos_norm @ pos_norm.T).numpy()   # cosine similarity between all patch positions

# Show as a 14×14 → 14×14 similarity heatmap (reshape for spatial interpretation)
# For each of the 196 positions, show its similarity to all other positions
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

# Show similarity from 8 selected reference patches to all 196 patches
reference_patches = [0, 6, 13, 98, 97, 99, 182, 195]
titles = ["top-left", "top-mid", "top-right", "centre-L",
          "centre", "centre-R", "bottom-L", "bottom-right"]

for ax, ref_idx, title in zip(axes.flat, reference_patches, titles):
    heatmap = sim[ref_idx].reshape(14, 14)
    im = ax.imshow(heatmap, cmap="hot", vmin=0, vmax=1)
    ax.set_title(f"Sim. from patch '{title}'\n(pos {ref_idx})")
    ax.axis("off")

plt.suptitle("ViT-B/16 positional embedding cosine similarities\n"
             "High similarity (yellow) = nearby in the 14×14 patch grid", y=1.02)
plt.tight_layout()
plt.show()

print("Observation: ViT learns that spatially adjacent patches have similar position embeddings,")
print("even though positions were only given 1-D integer indices during training.")

## 7 · Summary — PixelSmith v1

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v1 — Visual Frontend                                               │
│                                                                                │
│  (3, 224, 224) image tensor                 (from Ch.1)                       │
│      │                                                                         │
│      ▼  Conv2d(P=16, stride=16)                                               │
│  (196, 768) patch embeddings                                                   │
│      │                                                                         │
│      ▼  Prepend [CLS], add pos embeddings                                     │
│  (197, 768) sequence                                                           │
│      │                                                                         │
│      ▼  12 × Transformer encoder layers (MSA + MLP)                           │
│  (197, 768) encoded sequence                                                   │
│      │                                                                         │
│      ▼  Take CLS token (position 0)                                           │
│    (768,) image embedding  ← ready for CLIP alignment (Ch.3)                 │
└──────────────────────────────────────────────────────────────────────────────┘
```

**Next:** [CLIP.md](../ch03_clip/CLIP.md) — train a text encoder alongside this ViT with contrastive loss so that image and text embeddings live in the same space.